# scRareBench | High-Level Harmony | Dataset 5 — NYGC / Seurat v4 PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

**Dataset:** NYGC / Seurat v4 CITE-seq PBMC · **Method:** Harmony (`harmonypy==2.0.0`)

`load_dataset(5) → MethodSpec(Harmony) → benchmark_method() → scIB + rare-cell metrics → HTML + ZIP`

- scRareBench handles dataset preparation (including official Dataset 5 QC), registered rare-scenario annotations, evaluation, and report generation.
- Harmony uses **raw RNA counts in `adata.X`** and the registered batch column `orig.ident`. The biological labels are used **only** by the benchmark.
- Method-specific processing follows the Dataset 0 reference: **4,000 batch-aware Seurat-v3 HVGs → normalize → log1p → scale → 50 PCs → Harmony**.
- Runs three method seeds `[42, 123, 2026]` with a fixed benchmark seed of `42`.

Harmony runs on CPU; a **High-RAM Colab** runtime is recommended for this dataset. The scIB metrics use **JAX on CPU** to avoid a CUDA/cuSolver initialization failure seen in the `pcr_comparison` metric for this dataset. This does not change method or evaluation settings. **Start with Runtime → Restart session, then Run all**, because this setting must precede the first JAX import.

## 1. Install scRareBench and Harmony

In [ ]:
import os
# JAX is used internally by scIB. Keep it on CPU to avoid the Colab GPU
# cuSolver handle error during scIB PCR comparison. Must run before JAX loads.
os.environ["JAX_PLATFORMS"] = "cpu"

import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"  # scRareBench v0.10.6
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    f"git+https://github.com/amirhossein-alishahi/scRareBench_.git@{SCRAREBENCH_REVISION}",
])

METHOD_DEPENDENCIES = ("harmonypy==2.0.0",)

from scrarebench.runtime import setup_runtime

setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("harmonypy",),
)

## 2. Load Dataset 5

In [ ]:
from scrarebench import load_dataset

adata = load_dataset(5)
print(adata)

## 3. Configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42
OUTPUT_DIR = "/content/scrarebench_results/dataset5_harmony"

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "normalize_target_sum": 10_000.0,
    "scale_max_value": 10.0,
    "n_pcs": 50,
    "pca_solver": "arpack",
    "theta": 2.0,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 4,
    "epsilon_cluster": 1e-3,
    "epsilon_harmony": 1e-2,
    "alpha": 0.2,
    "batch_prop_cutoff": 1e-5,
    "ncores": 0,
    "verbose": True,
}

## 4. Harmony runner

This is the only method-specific implementation. HVG selection reads the canonical raw-count `X` without modifying the benchmark input. The runner supplies **only the batch column** to Harmony and returns aligned cell × PC embeddings.

In [ ]:
import harmonypy as hm
import numpy as np
import scanpy as sc

from scrarebench import MethodOutput, dataset_info


def run_harmony(method_adata, seed, config):
    batch_key = dataset_info(method_adata)["batch_key"]

    # Seurat-v3 HVGs are chosen from raw RNA counts in adata.X.
    hvg = sc.pp.highly_variable_genes(
        method_adata,
        layer=None,
        flavor="seurat_v3",
        n_top_genes=config["n_hvg"],
        batch_key=batch_key,
        span=config["hvg_span"],
        inplace=False,
    )
    work = method_adata[:, hvg["highly_variable"].to_numpy()].copy()

    sc.pp.normalize_total(work, target_sum=config["normalize_target_sum"])
    sc.pp.log1p(work)
    sc.pp.scale(work, max_value=config["scale_max_value"])
    sc.pp.pca(
        work,
        n_comps=config["n_pcs"],
        zero_center=True,
        svd_solver=config["pca_solver"],
        random_state=int(seed),
    )

    harmony = hm.run_harmony(
        work.obsm["X_pca"],
        work.obs[[batch_key]],
        batch_key,
        theta=config["theta"],
        lamb=config["lamb"],
        sigma=config["sigma"],
        nclust=config["nclust"],
        tau=config["tau"],
        block_size=config["block_size"],
        max_iter_harmony=config["max_iter_harmony"],
        max_iter_kmeans=config["max_iter_kmeans"],
        epsilon_cluster=config["epsilon_cluster"],
        epsilon_harmony=config["epsilon_harmony"],
        alpha=config["alpha"],
        batch_prop_cutoff=config["batch_prop_cutoff"],
        ncores=config["ncores"],
        verbose=config["verbose"],
        random_state=int(seed),
    )

    return MethodOutput(
        latent=np.asarray(harmony.Z_corr, dtype=np.float32),
        barcodes=work.obs_names.copy(),
        representation_key="X_pca_harmony",
    )

## 5. Run the High-Level benchmark

scIB runs JAX-based metrics on CPU to avoid the cuSolver error observed after Harmony finished. The method seeds, Harmony parameters, and benchmark metrics remain unchanged.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="Harmony",
    runner=run_harmony,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=False,
    overwrite=True,
)

## 6. Results and download

In [ ]:
from IPython.display import display
from google.colab import files
from pathlib import Path
import shutil

display(result.summary().round(5))
print("Multi-seed HTML:", result.report_path)
print("Multi-seed ZIP:", result.archive_path)

archive = Path("/content/Harmony_dataset5_results.zip")
shutil.copy2(result.archive_path, archive)
files.download(str(archive))